# Fake News Detection — NLP Case Study
This notebook prepares the ISOT Fake and Real News dataset, compares Multinomial Naive Bayes and Logistic Regression using TF-IDF, evaluates the selected model, and saves the trained pipeline and metrics.

> **Important:** This is a text-pattern classifier trained on historical dataset labels. It is not an independent fact-checking system.

## 1. Import libraries and load the dataset
The loader downloads `Fake.csv` and `True.csv` into `data/` the first time it runs.

In [ ]:
from src_data_loader import load_and_clean_dataset
import pandas as pd
data = load_and_clean_dataset()
data.head()

## 2. Explore the dataset


In [ ]:
print('Shape:', data.shape)
print('\nClass counts:')
print(data['label'].value_counts())
data.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
data['label'].value_counts().plot(kind='bar', title='News class distribution')
plt.xlabel('Class')
plt.ylabel('Number of articles')
plt.tight_layout()
plt.show()

## 3. Prepare training and testing data
An 80:20 stratified split preserves the class proportions in both subsets. TF-IDF is fitted inside each pipeline using training data only, avoiding test-set vocabulary leakage.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    data['content'], data['label'], test_size=0.20,
    random_state=42, stratify=data['label']
)
print('Training records:', len(X_train))
print('Testing records:', len(X_test))

## 4. Train and compare models
Both models use TF-IDF word unigrams and bigrams. Macro-averaged scores give equal weight to Fake and Real classes.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def make_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode',
            ngram_range=(1, 2), min_df=2, max_df=0.95,
            max_features=100000, sublinear_tf=True)),
        ('classifier', classifier)
    ])

models = {
    'Multinomial Naive Bayes': make_pipeline(MultinomialNB(alpha=0.5)),
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
}
results, trained_models = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        'Accuracy': accuracy_score(y_test, pred),
        'Macro precision': precision_score(y_test, pred, average='macro', zero_division=0),
        'Macro recall': recall_score(y_test, pred, average='macro', zero_division=0),
        'Macro F1': f1_score(y_test, pred, average='macro', zero_division=0)
    }
    trained_models[name] = model
pd.DataFrame(results).T

## 5. Select the best model and evaluate it
The model with the highest macro F1-score is selected.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
best_name = max(results, key=lambda name: results[name]['Macro F1'])
best_model = trained_models[best_name]
y_pred = best_model.predict(X_test)
print('Selected model:', best_name)
print(classification_report(y_test, y_pred, labels=['Fake', 'Real'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=['Fake', 'Real'], display_labels=['Fake', 'Real'], cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## 6. Save model and evaluation metrics
The saved pipeline contains both TF-IDF transformation and classifier, so the Streamlit application can predict directly from article text.

In [ ]:
from pathlib import Path
import json, joblib
ROOT = Path.cwd()
(ROOT / 'models').mkdir(exist_ok=True)
(ROOT / 'outputs').mkdir(exist_ok=True)
joblib.dump(best_model, ROOT / 'models' / 'fake_news_pipeline.joblib')
metrics = {
    'best_model': best_name,
    'dataset_records_after_cleaning': int(len(data)),
    'fake_records': int((data.label == 'Fake').sum()),
    'real_records': int((data.label == 'Real').sum()),
    'train_records': int(len(X_train)),
    'test_records': int(len(X_test)),
    'models': {name: {key.lower().replace(' ', '_'): float(value) for key, value in values.items()} for name, values in results.items()}
}
(ROOT / 'outputs' / 'metrics.json').write_text(json.dumps(metrics, indent=4), encoding='utf-8')
(ROOT / 'outputs' / 'classification_report.txt').write_text(classification_report(y_test, y_pred, labels=['Fake', 'Real'], zero_division=0), encoding='utf-8')
print('Saved model and evaluation outputs.')
print(json.dumps(metrics, indent=2))

## 7. Test a sample article
The output is a prediction based on learned dataset patterns, not a factual verdict.

In [ ]:
sample = 'Paste a complete news headline and article body here. The text should contain enough context for the model to analyse.'
print('Predicted class:', best_model.predict([sample])[0])
print('Class probabilities:', dict(zip(best_model.classes_, best_model.predict_proba([sample])[0].round(3))))